In [ ]:
#### Time-resolved spectra 3 peak deconvolution and peak intensity change
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob
import os
import re
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

# ==========================================
# 1. CONFIGURATION & LIMITS
# ==========================================
raman_folder = 'spectra_Anod-SS1'
range_min, range_max = 2500, 4000
anchor_zones = [(2500, 2550), (3950, 4000)]

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def extract_potential(filename):
    """
    Extracts numerical potential from filename. 
    Adjust the regex if your naming convention is different.
    Example: 'spectrum_-0.5V.txt' -> -0.5
    """
    match = re.search(r"([-+]?\d*\.\d+|\d+)", filename)
    return float(match.group()) if match else 0.0

files = sorted(glob.glob(os.path.join(raman_folder, "*.txt")), key=natural_sort_key)

# Data storage for the Trend Plot
results_list = []

# ==========================================
# 2. MODELS
# ==========================================
def gaussian(x, amp, ctr, sigma):
    return amp * np.exp(-(x - ctr)**2 / (2 * sigma**2))

def fit_model(x, a1, c1, s1, a2, c2, s2, a3, c3, s3):
    return gaussian(x, a1, c1, s1) + gaussian(x, a2, c2, s2) + gaussian(x, a3, c3, s3)

# ==========================================
# 3. MAIN PROCESSING LOOP
# ==========================================
# (Initial setup for waterfall plot)
test_df = pd.read_csv(files[0], sep='\s+', comment='#', header=None, encoding='latin1')
test_win = test_df[(test_df[0] >= range_min) & (test_df[0] <= range_max)]
raman_offset = (test_win[1].max() - test_win[1].min()) * 1.5 

fig, ax = plt.subplots(figsize=(12, 0.8 * len(files))) 
colors_map = plt.cm.plasma(np.linspace(0, 0.8, len(files)))

for i, file_path in enumerate(files):
    filename = os.path.basename(file_path).replace('.txt', '')
    potential = extract_potential(filename)
    offset = i * raman_offset
    
    try:
        df = pd.read_csv(file_path, sep='\s+', comment='#', header=None, encoding='latin1')
        df = df[(df[0] >= range_min) & (df[0] <= range_max)]
        wn, inty_raw = df[0].values, df[1].values

        # Smoothing & BG Correction
        inty_s = savgol_filter(inty_raw, window_length=31, polyorder=3)
        mask = np.zeros(len(wn), dtype=bool)
        for z_min, z_max in anchor_zones: mask |= (wn >= z_min) & (wn <= z_max)
        poly_coeffs = np.polyfit(wn[mask], inty_s[mask], 2) 
        inty_f = inty_s - np.polyval(poly_coeffs, wn)
        inty_f -= inty_f.min()
        
        # Peak Fitting
        max_h = np.max(inty_f)
        p1_z = (wn > 3230) & (wn < 3290); gc1 = wn[p1_z][np.argmax(inty_f[p1_z])] if any(p1_z) else 3260
        p2_z = (wn > 3404) & (wn < 3464); gc2 = wn[p2_z][np.argmax(inty_f[p2_z])] if any(p2_z) else 3434
        p3_z = (wn > 3587) & (wn < 3647); gc3 = wn[p3_z][np.argmax(inty_f[p3_z])] if any(p3_z) else 3617

        guesses = [max_h*0.7, gc1, 15, max_h, gc2, 15, max_h*0.5, gc3, 15]
        low_b = [0, 3230, 5, 0, 3404, 5, 0, 3587, 5]
        upp_b = [np.inf, 3290, 60, np.inf, 3464, 60, np.inf, 3647, 60]

        popt, _ = curve_fit(fit_model, wn, inty_f, p0=guesses, bounds=(low_b, upp_b))
        
        # SAVE RESULTS: Storing Amplitudes (popt[0], popt[3], popt[6])
        results_list.append({
            'potential': potential,
            'Peak1_Int': popt[0],
            'Peak2_Int': popt[3],
            'Peak3_Int': popt[6]
        })

        # Waterfall Plotting (same as your original)
        peaks = [gaussian(wn, *popt[0:3]), gaussian(wn, *popt[3:6]), gaussian(wn, *popt[6:9])]
        ax.plot(wn, sum(peaks) + offset, color=colors_map[i], lw=1.5)

        # Plot Filled Areas
        for j, p in enumerate(peaks):
            ax.fill_between(wn, offset, p + offset, color=colors_map[i], alpha=0.15 + (j*0.1))

        # NEW: Label with Potential instead of Filename
        # We place it to the right of the spectrum
        ax.text(range_max + 15, offset + (raman_offset * 0.1), f"{potential:.2f} mV", 
                fontsize=10, va='center', fontweight='bold', color=colors_map[i])

       # Center Labels for all 3 peaks
        label_style = dict(size=7, color='black', ha='center', weight='bold',
                               bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
        
        for j in [1, 4, 7]: # Indices of centers in popt
            ax.text(popt[j], popt[j-1] + offset + (raman_offset*0.05), f'{popt[j]:.0f}', **label_style)  


    except Exception as e:
        print(f"Error on {filename}: {e}")

# Formatting Waterfall
ax.set_xlim(range_min, range_max + 180) # Increased margin for labels
ax.set_xlabel("Wavenumber ($cm^{-1}$)", fontsize=13, fontweight='bold')
ax.set_yticks([]); ax.spines['left'].set_visible(False)
ax.grid(axis='x', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

# ==========================================
# 5. CHRONOLOGICAL INTENSITY PLOT (Fancy Colors)
# ==========================================
res_df = pd.DataFrame(results_list)
x_axis = res_df.index 

# Define a "Fancy" color palette (Professional Hex Codes)
# Deep Purple, Vibrant Orange, Teal Blue
fancy_colors = ['#5E2CA5', '#FF725C', '#00A19D'] 

plt.figure(figsize=(10, 6))

# Plotting the peaks with fancy colors and the legend in top right
plt.plot(x_axis, res_df['Peak1_Int'], 'o-', color=fancy_colors[0], 
         label='Peak ~3260 $cm^{-1}$', markersize=6, linewidth=2, alpha=0.9)

plt.plot(x_axis, res_df['Peak2_Int'], 's-', color=fancy_colors[1], 
         label='Peak ~3434 $cm^{-1}$', markersize=6, linewidth=2, alpha=0.9)

plt.plot(x_axis, res_df['Peak3_Int'], '^-', color=fancy_colors[2], 
         label='Peak ~3617 $cm^{-1}$', markersize=6, linewidth=2, alpha=0.9)

# --- X-AXIS FORMATTING (Potentials) ---
plt.xticks(
    x_axis, 
    [f"{p:.1f}" for p in res_df['potential']], 
    rotation=45, 
    fontsize=9
)

# --- LEGEND & LABELS ---
# loc='upper right' moves the label to the top right corner
plt.legend(loc='upper right', frameon=True, facecolor='white', framealpha=1, edgecolor='lightgray')

plt.xlabel('Potential (mV)', fontsize=12, fontweight='bold')
plt.ylabel('Deconvoluted Intensity (a.u.)', fontsize=12, fontweight='bold')
plt.title('Intensity Evolution across OER Potentials', fontsize=14, pad=15)

# Increase the top limit by 30% to make room
current_ylim = plt.gca().get_ylim()
plt.ylim(current_ylim[0], current_ylim[1] * 1.2)

# Adding a light grid for a modern look
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()